# 🦟 Análisis Exploratorio de Datos — Dengue en Ecuador

Este notebook realiza un análisis exploratorio completo del dataset sintético de dengue
generado para el Hackathon AironTech.

**Objetivos:**
- Comprender la distribución de variables
- Identificar correlaciones relevantes
- Analizar patrones temporales y geográficos
- Formular hipótesis para el modelado

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='husl')

print("Librerías cargadas correctamente.")

## 1. Carga y Generación de Datos

Generamos el dataset sintético si no existe, o lo cargamos desde disco.

In [ ]:
from pathlib import Path

RAW_PATH = Path('../data/raw/dengue_ecuador_sintetico.csv')

if not RAW_PATH.exists():
    print("Generando datos sintéticos...")
    from src.data.generate_synthetic_data import generate_synthetic_data
    df = generate_synthetic_data()
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(RAW_PATH, index=False)
    print(f"Dataset guardado: {RAW_PATH}")
else:
    df = pd.read_csv(RAW_PATH)
    print(f"Dataset cargado: {df.shape[0]:,} registros")

df.head()

## 2. Descripción General del Dataset

In [ ]:
print("=== Información General ===")
print(f"Filas      : {df.shape[0]:,}")
print(f"Columnas   : {df.shape[1]}")
print(f"Años       : {df['year'].min()} - {df['year'].max()}")
print(f"Cantones   : {df['canton_name'].nunique()}")
print(f"Provincias : {df['provincia'].nunique()}")
print()
df.dtypes

In [ ]:
df.describe(include='all').T

## 3. Análisis de Valores Faltantes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Faltantes': missing, 'Porcentaje (%)': missing_pct})
missing_df = missing_df[missing_df['Faltantes'] > 0]

if missing_df.empty:
    print("✅ No hay valores faltantes en el dataset.")
else:
    print(missing_df)

## 4. Distribución de Variables Numéricas

In [ ]:
num_cols = ['temperatura_promedio', 'precipitacion_mm', 'indice_aedes',
            'altitud_msnm', 'densidad_poblacional', 'cobertura_salud', 'casos_dengue']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel('Frecuencia')
    axes[i].grid(axis='y', alpha=0.3)

# Ocultar el último subplot vacío
axes[-1].set_visible(False)

plt.suptitle('Distribución de Variables Numéricas', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/eda_distribuciones.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figura guardada en reports/eda_distribuciones.png")

## 5. Variable Objetivo: Casos de Dengue y Nivel de Riesgo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de casos (log scale)
axes[0].hist(df['casos_dengue'][df['casos_dengue'] > 0], bins=50,
             color='coral', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Casos de Dengue (semanas con casos > 0)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Casos de Dengue')
axes[0].set_yscale('log')

# Nivel de riesgo
risk_order = ['bajo', 'medio', 'alto', 'muy_alto']
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
counts = df['nivel_riesgo'].value_counts().reindex(risk_order)
axes[1].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[1].set_xlabel('Nivel de Riesgo')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución del Nivel de Riesgo')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 500, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/eda_target.png', dpi=120, bbox_inches='tight')
plt.show()

print("Proporción de brotes (brote=1):", f"{df['brote'].mean():.2%}")

## 6. Análisis de Correlación

In [ ]:
corr_cols = ['temperatura_promedio', 'precipitacion_mm', 'indice_aedes',
             'altitud_msnm', 'densidad_poblacional', 'cobertura_salud',
             'casos_dengue', 'brote']

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=ax, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlación', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/eda_correlacion.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nCorrelación con casos_dengue:")
print(corr_matrix['casos_dengue'].sort_values(ascending=False))

## 7. Análisis de Series Temporales

In [ ]:
# Casos totales por semana epidemiológica y año
ts = df.groupby(['year', 'semana_epidemiologica'])['casos_dengue'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
for yr, grp in ts.groupby('year'):
    ax.plot(grp['semana_epidemiologica'], grp['casos_dengue'],
            label=str(yr), linewidth=2, marker='o', markersize=3, alpha=0.8)

ax.set_xlabel('Semana Epidemiológica')
ax.set_ylabel('Casos Totales')
ax.set_title('Evolución Anual de Casos de Dengue en Ecuador (2018-2023)')
ax.legend(title='Año', ncol=3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/eda_serie_temporal.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Análisis Geográfico por Provincia

In [ ]:
prov_casos = (df.groupby('provincia')['casos_dengue']
              .sum()
              .sort_values(ascending=True))

fig, ax = plt.subplots(figsize=(10, 9))
colors_bar = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(prov_casos)))
ax.barh(prov_casos.index, prov_casos.values, color=colors_bar, edgecolor='white')
ax.set_xlabel('Total de Casos de Dengue (2018-2023)')
ax.set_title('Total de Casos de Dengue por Provincia')
for i, v in enumerate(prov_casos.values):
    ax.text(v + 100, i, f'{v:,.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../reports/eda_provincias.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Relación Altitud y Temperatura con Casos de Dengue

In [ ]:
sample = df.sample(min(5000, len(df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Altitud vs casos
axes[0].scatter(sample['altitud_msnm'], sample['casos_dengue'],
                alpha=0.3, s=8, color='steelblue')
axes[0].set_xlabel('Altitud (msnm)')
axes[0].set_ylabel('Casos de Dengue')
axes[0].set_title('Altitud vs Casos de Dengue')
axes[0].grid(alpha=0.2)

# Temperatura vs casos
sc = axes[1].scatter(sample['temperatura_promedio'], sample['casos_dengue'],
                     c=sample['indice_aedes'], cmap='YlOrRd',
                     alpha=0.4, s=8)
plt.colorbar(sc, ax=axes[1], label='Índice Aedes')
axes[1].set_xlabel('Temperatura Promedio (°C)')
axes[1].set_ylabel('Casos de Dengue')
axes[1].set_title('Temperatura vs Casos (color = Índice Aedes)')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../reports/eda_altitud_temperatura.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Hallazgos Principales

### Variables con mayor influencia en los casos de dengue:

| Variable | Relación | Justificación |
|----------|----------|---------------|
| `altitud_msnm` | 🔴 Negativa | El *Aedes aegypti* no sobrevive a >2000 msnm |
| `temperatura_promedio` | 🟢 Positiva | Temperaturas altas aceleran el ciclo del vector |
| `precipitacion_mm` | 🟢 Positiva | Las lluvias crean criaderos (agua estancada) |
| `indice_aedes` | 🟢 Positiva fuerte | Densidad directa del mosquito transmisor |
| `cobertura_salud` | 🔴 Negativa | Mejor cobertura reduce casos reportados |
| `densidad_poblacional` | 🟢 Positiva | Más humanos = más hospes disponibles |

### Patrones temporales:
- Los picos de dengue coinciden con la **temporada lluviosa** (semanas 1-20 en la Costa)
- Se observa una **tendencia creciente** de 2018 a 2023
- Las provincias costeras (Guayas, Manabí, Los Ríos) concentran la mayor carga

### Distribución de riesgo:
- La mayoría de registros son de nivel **"bajo"** (semanas sin casos)
- Los brotes están altamente concentrados en zonas costeras y amazónicas de baja altitud